In [ ]:
!pip install biopython pandas numpy scikit-learn seaborn

from collections import Counter
import matplotlib.pyplot as plt
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# ================================
# 0. Constant Definitions
# ================================

AAS = "ACDEFGHIKLMNPQRSTVWY"
SW_WINDOWS = [20, 40]

AA_GROUPS = {
    "KR": "KR",
    "KRH": "KRH",
    "ED": "ED",
}

AA_GROUPS_EXT = {
    "STNQCH": "STNQCH",
    "ILMV": "ILMV",
    "FWY": "FWY",
}

# Merge both group sets for AAC computation
AAC_GROUPS_ALL = {**AA_GROUPS_EXT, **AA_GROUPS}


def clean_seq(seq):
    return seq.replace("U", "C")


# ================================
# 1. Load High / Mid / Low Group IDs
# ================================

df_high = pd.read_excel("High_group_ID.xlsx")
df_mid = pd.read_excel("Mid_group_ID.xlsx")
df_low = pd.read_excel("Low_group_ID.xlsx")

df_high["Class3"] = "High"
df_mid["Class3"] = "Mid"
df_low["Class3"] = "Low"

df_class = pd.concat([df_high, df_mid, df_low], ignore_index=True)

# ================================
# 2. Sequence Processing
# ================================

seqs = {
    rec.id: str(rec.seq)
    for rec in SeqIO.parse("ALL_protein_sequences.fasta", "fasta")
}

df_seq = pd.DataFrame(seqs.items(), columns=["ID", "Sequence"])

df = df_class.merge(df_seq, on="ID")
df["Sequence"] = df["Sequence"].map(clean_seq)

df = df.reset_index(drop=True)

# ================================
# 3. Binary Labeling (High / Low)
# ================================

df_binary = df[df["Class3"].isin(["High", "Low"])].copy()
df_binary["Class2"] = df_binary["Class3"].map({"Low": 0, "High": 1})

# ================================
# 4. Physicochemical Features + AAC (+ Group AAC)
# ================================


def physchem_features(seq):
    prot = ProteinAnalysis(seq)
    aa = prot.count_amino_acids()
    L = len(seq)

    # --- Basic Physicochemical Properties ---
    base = [
        prot.molecular_weight(),
        prot.gravy(),
        prot.charge_at_pH(7.5),
        prot.charge_at_pH(6.8),
        prot.charge_at_pH(6.8) - prot.charge_at_pH(7.5),
        prot.isoelectric_point(),
    ]

    # --- Single Amino Acid Composition (AAC) ---
    aac = [aa[a] / L for a in AAS]

    # --- Grouped Amino Acid Composition (EXT + Charge) ---
    aac_group = [
        sum(aa[a] for a in group) / L for group in AAC_GROUPS_ALL.values()
    ]

    return base + aac + aac_group


pc_cols = (
    ["MW", "GRAVY", "Charge_7.5", "Charge_6.8", "DeltaCharge_6.8_7.5", "pI"]
    + [f"AAC_{a}" for a in AAS]
    + [f"AACgroup_{k}" for k in AAC_GROUPS_ALL]
)

df_pc_all = pd.DataFrame(
    [physchem_features(s) for s in df["Sequence"]], columns=pc_cols
)

# ================================
# 5. Terminal Bias Index
# ================================


def termial_bias(seq, aa_set):
    L = len(seq)
    positions = [
        abs((i + 0.5) / L - 0.5) for i, a in enumerate(seq) if a in aa_set
    ]
    return np.mean(positions) if positions else 0.0


df_term_aa = pd.DataFrame(
    [[termial_bias(s, a) for a in AAS] for s in df["Sequence"]],
    columns=[f"TermBias_{a}" for a in AAS],
)

df_term_group = pd.DataFrame(
    [
        [termial_bias(s, g) for g in AA_GROUPS_EXT.values()]
        for s in df["Sequence"]
    ],
    columns=[f"TermBias_{k}" for k in AA_GROUPS_EXT],
)

df_term_charge = pd.DataFrame(
    [[termial_bias(s, g) for g in AA_GROUPS.values()] for s in df["Sequence"]],
    columns=[f"TermBias_{k}" for k in AA_GROUPS],
)


# ================================
# 6. Sliding Window Features (Theoretical Variance Normalization)
# ================================


def sliding_window_var_norm_aa(seq, aas):
    out = []
    L = len(seq)
    cnt_all = Counter(seq)

    for w in SW_WINDOWS:
        for a in aas:
            p = cnt_all[a] / L
            vals = []

            for i in range(L - w + 1):
                sub = seq[i : i + w]
                vals.append(sub.count(a) / w)

            var_obs = np.var(vals) if vals else 0.0
            var_exp = p * (1 - p) / w
            out.append(var_obs / (var_exp + 1e-6))
    return out


df_sw_aa = pd.DataFrame(
    [sliding_window_var_norm_aa(s, AAS) for s in df["Sequence"]],
    columns=[f"SW{w}_Var_{a}" for w in SW_WINDOWS for a in AAS],
)


def sliding_window_var_norm_group(seq, groups):
    out = []
    L = len(seq)
    cnt_all = Counter(seq)

    for w in SW_WINDOWS:
        for k, g in groups.items():
            p = sum(cnt_all[a] for a in g) / L
            vals = []

            for i in range(L - w + 1):
                sub = seq[i : i + w]
                vals.append(sum(sub.count(a) for a in g) / w)

            var_obs = np.var(vals) if vals else 0.0
            var_exp = p * (1 - p) / w
            out.append(var_obs / (var_exp + 1e-6))
    return out


df_sw_group = pd.DataFrame(
    [sliding_window_var_norm_group(s, AA_GROUPS_EXT) for s in df["Sequence"]],
    columns=[f"SW{w}_Var_{k}" for w in SW_WINDOWS for k in AA_GROUPS_EXT],
)

df_sw_charge = pd.DataFrame(
    [sliding_window_var_norm_group(s, AA_GROUPS) for s in df["Sequence"]],
    columns=[f"SW{w}_Var_{k}" for w in SW_WINDOWS for k in AA_GROUPS],
)

# ================================
# 7. k-mer (k=2): Symmetrical Pairing & Mean Frequency Filtering
# ================================

KMER_K = 2

# Combine AB and BA pairs symmetrically (e.g., AA, AC, AD, ..., YY)
KMER2_SYM_LIST = sorted(
    set("".join(sorted(a + b)) for a in AAS for b in AAS)
)


def kmer2_symmetric_features(seq):
    L = len(seq)

    counts = Counter(
        "".join(sorted(seq[i : i + 2])) for i in range(L - 1)
    )

    denom = max(L - 1, 1)
    return [counts.get(km, 0) / denom for km in KMER2_SYM_LIST]


df_kmer2_sym = pd.DataFrame(
    [kmer2_symmetric_features(s) for s in df["Sequence"]],
    columns=[f"Pair_{km}" for km in KMER2_SYM_LIST],
)

KMER_MEAN_FREQ_TH = 1e-3

kmer_mean_freq = df_kmer2_sym.mean()
keep_kmers = kmer_mean_freq[kmer_mean_freq >= KMER_MEAN_FREQ_TH].index
df_kmer2_sym_filt = df_kmer2_sym[keep_kmers]


# ================================
# 8. Final Feature Matrix
# ================================

df_features_all = pd.concat(
    [
        df_pc_all,
        df_term_aa,
        df_term_group,
        df_term_charge,
        df_sw_aa,
        df_sw_group,
        df_sw_charge,
        df_kmer2_sym_filt,
    ],
    axis=1,
)


# ================================
# 9. Random Forest Classification (High vs Low)
# ================================

X = df_features_all.loc[df_binary.index]
y = df_binary["Class2"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

print("--- Classification Report ---")
print(classification_report(y_test, clf.predict(X_test)))

# ================================
# 10. Feature Importance -> Top 100
# ================================

importances = pd.Series(clf.feature_importances_, index=df_features_all.columns)

top100 = importances.sort_values(ascending=False).head(100)
print("\n--- Top 100 Feature Importances ---")
print(top100)

# ================================
# 11. Heatmap of Mean Feature Values across High / Mid / Low Groups
# ================================

df_group = pd.concat([df["Class3"], df_features_all], axis=1)

mean_table = df_group.groupby("Class3")[top100.index].mean()

# Explicitly specify class display order
class_order = ["High", "Mid", "Low"]
mean_table = mean_table.reindex(class_order)

# Standardize values per feature (z-score)
mean_table_std = (mean_table - mean_table.mean()) / mean_table.std()

plt.figure(figsize=(50, 10))
sns.heatmap(mean_table_std, cmap="coolwarm", center=0)
plt.title("Top100 Features × High / Mid / Low (Standardized)")
plt.xlabel("Features")
plt.ylabel("Class")
plt.yticks(rotation=0)
plt.show()

In [ ]:
# ================================
# Cross-Validation (Stratified)
# ================================

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

clf_cv = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# ---- Accuracy CV ----
acc_scores = cross_val_score(
    clf_cv, X, y,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("\n[CV Accuracy]")
print("Scores:", acc_scores)
print(f"Mean ACC = {acc_scores.mean():.4f} ± {acc_scores.std():.4f}")


# ---- AUC CV ----
auc_scores = cross_val_score(
    clf_cv, X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("\n[CV AUC]")
print("Scores:", auc_scores)
print(f"Mean AUC = {auc_scores.mean():.4f} ± {auc_scores.std():.4f}")


# ---- CV Predicted Probabilities ----
probs_cv = cross_val_predict(
    clf_cv, X, y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

pred_cv = (probs_cv > 0.5).astype(int)


# ---- CV classification report ----
from sklearn.metrics import classification_report

print("\n[CV Classification Report]")
print(classification_report(y, pred_cv))


# ================================
# CV Feature Importance
# ================================

importances_all = []

for train_idx, test_idx in cv.split(X, y):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

    clf_cv.fit(Xtr, ytr)
    importances_all.append(clf_cv.feature_importances_)

imp_mean = pd.Series(
    np.mean(importances_all, axis=0),
    index=X.columns
).sort_values(ascending=False)

imp_std = pd.Series(
    np.std(importances_all, axis=0),
    index=X.columns
)

top100_cv = imp_mean.head(100)

print("\n[Top100 CV Feature Importance]")
print(top100_cv)


# ================================
# CV Top100 Heatmap (High/Mid/Low)
# ================================

df_group_cv = pd.concat([df["Class3"], df_features_all], axis=1)

mean_table_cv = df_group_cv.groupby("Class3")[top100_cv.index].mean()
mean_table_cv = mean_table_cv.reindex(class_order)

mean_table_cv_std = (
    mean_table_cv - mean_table_cv.mean()
) / mean_table_cv.std()

plt.figure(figsize=(50, 10))
sns.heatmap(
    mean_table_cv_std,
    cmap="coolwarm",
    center=0
)

plt.title("Top100 CV Features × High / Mid / Low (Standardized)")
plt.xlabel("Features")
plt.ylabel("Class")
plt.yticks(rotation=0)
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(9.5, 6))
ax = plt.gca()

for spine in ax.spines.values():
    spine.set_linewidth(3.0)

ax.tick_params(direction='out', length=15, width=2.5, colors='black')

tprs = []
aucs = []
mean_fpr = np.linspace(0, 1, 100)

fold_colors = ['tomato', 'tomato', 'tomato', 'tomato', 'tomato']

chance_color = 'black'
mean_roc_color = 'blue'

for i, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
    Xte, yte = X.iloc[test_idx], y.iloc[test_idx]

    clf_cv.fit(Xtr, ytr)
    viz_probs = clf_cv.predict_proba(Xte)[:, 1]

    fpr, tpr, thresholds = roc_curve(yte, viz_probs)
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)

    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)

    plt.plot(fpr, tpr, linestyle=':', color=fold_colors[i], lw=3.0, alpha=0.8,
             label=f'Fold {i+1} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', lw=1.5, color=chance_color, label='Chance (AUC = 0.50)', alpha=.8)

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(aucs)

plt.plot(mean_fpr, mean_tpr, color=mean_roc_color, linestyle='-',
         label=f'Mean ROC (AUC = {mean_auc:.3f} $\pm$ {std_auc:.3f})',
         lw=5.0, alpha=.9)

plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)

plt.grid(False)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, frameon=True)

plt.tight_layout()

plt.show()

In [ ]:
!pip install shap

import shap
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ================================
# SHAP Analysis - Customized Version
# ================================

explainer = shap.TreeExplainer(clf)

shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_values_high = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_values_high = shap_values[:, :, 1]
else:
    shap_values_high = shap_values

# --- Visualization: Summary Plot (Beeswarm) ---
my_cmap = cm.coolwarm

fig, ax = plt.figure(figsize=(10, 8)), plt.gca()

shap.summary_plot(
    shap_values_high,
    X_test,
    max_display=30,
    cmap=my_cmap,
    show=False
)

# --- Override plot appearance settings for customization ---
for spine in ax.spines.values():
    spine.set_linewidth(2.0)

ax.spines['left'].set_visible(True)
ax.spines['left'].set_color('black')

ax.tick_params(axis='x', direction='out', length=9, width=1.5, colors='black', labelsize=10)

ax.yaxis.set_ticks_position('left')
ax.tick_params(axis='y', direction='out', length=6, width=1.5, colors='black', labelsize=10)

plt.title("SHAP Summary Plot (Top 30 Features for High Class)", fontsize=14, pad=20)

plt.tight_layout()

for object_ax in fig.axes:
    if object_ax is not ax:
        pos = object_ax.get_position()

        shrink_factor = 0.5

        new_height = pos.height * shrink_factor
        new_y0 = pos.y0 + (pos.height - new_height) / 2

        object_ax.set_position([pos.x0, new_y0, pos.width, new_height])


plt.show()